# Our approach — GPT-OSS 120B on COM2SENSE (474 claims, earlier sample from the training split)

Facts written by the model, PRO/CON arguments, relation audit, bipolar framework, extensions and decision rules, run in batches. Results: credulous 74.3, skeptical 73.2, net support 75.3, 27.6% of claims with more than one extension.

Run on Kaggle; the API key is read from Kaggle secrets (`OPENROUTER_API_KEY`). Outputs are kept so the reported numbers can be checked.

In [2]:
!pip install -q openai datasets scikit-learn
import os, re, json, time, pickle, random
from pathlib import Path
from itertools import combinations
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from datasets import load_dataset

WORK_DIR = Path("/kaggle/working/gptoss_com2sense"); WORK_DIR.mkdir(parents=True, exist_ok=True)
client = OpenAI(api_key=UserSecretsClient().get_secret("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")
MODEL_NAME = "openai/gpt-oss-120b"

ds = load_dataset("tasksource/com2sense", split="train")
df = ds.to_pandas()
df["gold"] = df["label"].apply(lambda x: 1 if str(x).strip().lower()=="true" else 0)
df["claim"] = df["sent"].astype(str)
true_ex = df[df["gold"]==1].sample(250, random_state=42)
false_ex = df[df["gold"]==0].sample(250, random_state=42)
baf_df = pd.concat([true_ex, false_ex]).sample(frac=1, random_state=42).reset_index(drop=True)
baf_df["example_index"] = range(len(baf_df))
print("Loaded COM2SENSE:", len(baf_df), "| True:", baf_df["gold"].sum(), "| First:", baf_df.iloc[0]["claim"][:70])

BAF_CONFIG = {
    "n_pro_candidates": 5, "n_con_candidates": 5, "max_pro_keep": 4, "max_con_keep": 4,
    "min_relevance": 0.60, "min_grounding": 0.55, "min_clarity": 0.60, "min_quality": 0.58,
    "min_target_match": 0.65, "min_grounding_edge": 0.55, "min_relation_score": 0.68,
    "max_conflict_pairs": 3, "max_claim_attacks": 2, "audit_batch_size": 16,
    "generation_temperature": 0.15, "audit_temperature": 0.0, "max_retries": 5,
}
def clip01(v):
    try: return max(0.0,min(1.0,float(v)))
    except: return 0.0
def extract_json_object(text):
    text=str(text or "").strip(); text=re.sub(r"```(?:json)?","",text,flags=re.IGNORECASE).strip()
    try: return json.loads(text)
    except json.JSONDecodeError: pass
    s=text.find("{"); e=text.rfind("}")
    if s>=0 and e>s: return json.loads(text[s:e+1])
    raise ValueError("No JSON")
def deepseek_json_call(prompt, temperature, max_tokens, retries=5):
    last=None
    for a in range(1,retries+1):
        try:
            resp=client.chat.completions.create(model=MODEL_NAME,
                messages=[{"role":"system","content":"You are an expert in claim verification, formal argumentation and strict JSON output."},
                          {"role":"user","content":prompt}],
                temperature=temperature,max_tokens=max_tokens)
            t=resp.choices[0].message.content
            if not t: raise ValueError("empty")
            return extract_json_object(t)
        except Exception as e:
            last=e
            if "rate" in str(e).lower() or "429" in str(e): time.sleep(5); continue
            if a<retries: time.sleep(3*a)
    raise RuntimeError(f"failed: {last}")
def generate_facts(claim):
    prompt = f"""You are building atomic factual building blocks.
For the claim below, produce 3-5 SHORT, SEPARATE factual statements. Each must be:
- A single atomic fact about ONE concept in the claim.
- General commonsense knowledge, stated on its own.
- NOT a comparison, ranking, or conclusion. NOT about whether the claim is true or false.
Claim: "{claim}"
Return 3-5 atomic facts, one per line. Do not conclude."""
    for _ in range(5):
        try:
            resp=client.chat.completions.create(model=MODEL_NAME,
                messages=[{"role":"system","content":"You output atomic facts. You never conclude."},
                          {"role":"user","content":prompt}], temperature=0.3, max_tokens=1200)
            return (resp.choices[0].message.content or "").strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e): time.sleep(5); continue
            return ""
    return ""
print("✅ CELL 1 ready")

Loaded COM2SENSE: 500 | True: 250 | First: Tim was happy that November was coming up since he'd have a lot of tim
✅ CELL 1 ready


In [4]:
def generate_and_score_arguments(claim, facts_text, config=BAF_CONFIG):
    prompt = f"""
Construct candidate arguments for a Bipolar Argumentation Framework used for claim verification.

CLAIM:
{claim}

AVAILABLE FACTS:
{facts_text}

First determine the intended meaning of the claim, then reason carefully from facts and commonsense.

Generate up to {config["n_pro_candidates"]} PRO candidates and up to {config["n_con_candidates"]} CON candidates.

PRO: a specific, grounded reason the claim is TRUE.
CON: a specific, grounded reason the claim is FALSE.

CRITICAL rules for CON arguments (attacks):
- Only generate a CON if there is a GENUINE, SPECIFIC, STRONG reason the claim is false.
- Do NOT generate speculative CONs like "maybe not always", "it might not", "in some cases".
- Do NOT generate a CON that merely raises a weak possibility; it must give a real reason the claim is false.
- If the claim is clearly true and there is no strong objection, generate ZERO CON arguments.

CRITICAL rules for PRO arguments:
- Only generate a PRO if there is genuine support. If the claim is clearly false, generate ZERO PRO arguments.
- Do NOT generate circular or vague arguments.

General:
- Each argument must be specific and grounded, under 30 words.
- Do not output the final TRUE/FALSE label.

Score each candidate 0 to 1 for: relevance; grounding; clarity; overall_quality.
Score speculative, weak, vague, or circular arguments BELOW 0.4 on overall_quality.

Return valid JSON only:
{{
  "interpreted_claim": "brief meaning",
  "pro": [{{"text":"...","relevance":0.0,"grounding":0.0,"clarity":0.0,"overall_quality":0.0}}],
  "con": [{{"text":"...","relevance":0.0,"grounding":0.0,"clarity":0.0,"overall_quality":0.0}}]
}}
""".strip()
    raw=deepseek_json_call(prompt,config["generation_temperature"],2600,config["max_retries"])
    out={"interpreted_claim":str(raw.get("interpreted_claim","")).strip(),"pro":[],"con":[]}
    for role in ("pro","con"):
        for e in (raw.get(role,[]) if isinstance(raw.get(role,[]),list) else []):
            if not isinstance(e,dict): continue
            t=str(e.get("text","")).strip()
            if not t: continue
            out[role].append({"text":t,"relevance":clip01(e.get("relevance")),"grounding":clip01(e.get("grounding")),"clarity":clip01(e.get("clarity")),"overall_quality":clip01(e.get("overall_quality"))})
    return out

def select_arguments(gen, config=BAF_CONFIG):
    nodes={"CLAIM":{"role":"CLAIM","text":""}}; scored=[]
    for src,role,pre,mx in [("pro","PRO","P",config["max_pro_keep"]),("con","CON","N",config["max_con_keep"])]:
        cands=[]
        for it in gen.get(src,[]):
            rl,g=clip01(it.get("relevance")),clip01(it.get("grounding")); cl,q=clip01(it.get("clarity")),clip01(it.get("overall_quality"))
            rank=0.30*rl+0.30*g+0.15*cl+0.25*q; sc={**it,"role":role,"rank_score":rank}; scored.append(sc)
            if rl>=config["min_relevance"] and g>=config["min_grounding"] and cl>=config["min_clarity"] and q>=config["min_quality"]: cands.append(sc)
        cands.sort(key=lambda x:x["rank_score"],reverse=True)
        for n,it in enumerate(cands[:mx],1):
            nodes[f"{pre}{n}"]={"role":role,"text":it["text"],"relevance":it["relevance"],"grounding":it["grounding"],"clarity":it["clarity"],"overall_quality":it["overall_quality"],"rank_score":it["rank_score"]}
    return {"nodes":nodes,"all_scored":scored}

def build_relation_candidates(nodes):
    pro=[n for n,i in nodes.items() if i["role"]=="PRO"]; con=[n for n,i in nodes.items() if i["role"]=="CON"]; c=[]
    for p in pro: c.append({"source":p,"target":"CLAIM","allowed":["SUPPORT","NONE"]})
    for n in con: c.append({"source":n,"target":"CLAIM","allowed":["ATTACK","NONE"]})
    for p in pro:
        for n in con:
            c.append({"source":p,"target":n,"allowed":["ATTACK","NONE"]}); c.append({"source":n,"target":p,"allowed":["ATTACK","NONE"]})
    return c
print("✅ CELL 2 ready")

✅ CELL 2 ready


In [5]:
def audit_baf_relations(claim, facts_text, nodes, config=BAF_CONFIG, batch_size=16):
    cands=build_relation_candidates(nodes)
    at="\n".join(f"{nid} [{i['role']}]: {i['text']}" for nid,i in nodes.items() if nid!="CLAIM")
    aud=[]
    for bs in range(0,len(cands),batch_size):
        b=cands[bs:bs+batch_size]
        pb="\n".join(f"{it['source']} -> {it['target']} | allowed={it['allowed']}" for it in b)
        prompt=f"""Audit candidate relations for a Bipolar Argumentation Framework.

CLAIM:
{claim}
FACTS:
{facts_text}
ARGUMENTS:
{at}

For every candidate, choose exactly one allowed type.
SUPPORT: source gives a meaningful direct reason to accept target.
ATTACK: source directly contradicts, rebuts or undermines target.
NONE: no meaningful direct relation.
Use ATTACK for CON->CLAIM only if the CON gives a genuine reason the claim is false or doubtful.
Use NONE when the CON concedes the claim or agrees.
Keep every reason at eight words or fewer.
For ATTACK/SUPPORT scores normally exceed 0.50; for NONE normally below 0.50.

CANDIDATES:
{pb}

Return valid JSON only:
{{"relations":[{{"source":"P1","target":"CLAIM","type":"SUPPORT","target_match":0.9,"grounding":0.9,"relation_score":0.9,"reason":"..."}}]}}"""
        raw=deepseek_json_call(prompt,config["audit_temperature"],2800,config["max_retries"])
        exp={(it["source"],it["target"]) for it in b}
        allow={(it["source"],it["target"]):set(it["allowed"]) for it in b}
        ret={}
        for r in (raw.get("relations",[]) if isinstance(raw.get("relations",[]),list) else []):
            if not isinstance(r,dict): continue
            s,t=str(r.get("source","")).strip(),str(r.get("target","")).strip()
            typ=str(r.get("type","NONE")).strip().upper(); pr=(s,t)
            if pr not in exp or typ not in allow[pr]: continue
            ret[pr]={"source":s,"target":t,"type":typ,"target_match":clip01(r.get("target_match")),"grounding":clip01(r.get("grounding")),"relation_score":clip01(r.get("relation_score")),"reason":str(r.get("reason","")).strip()}
        for it in b:
            pr=(it["source"],it["target"])
            if pr not in ret:
                ret[pr]={"source":pr[0],"target":pr[1],"type":"NONE","target_match":0.0,"grounding":0.0,"relation_score":0.0,"reason":"omitted"}
        aud.extend(ret.values())
    return {"all_audited_relations":aud}
print("✅ CELL 3 ready")

✅ CELL 3 ready


In [6]:
def relation_passes(r,config=BAF_CONFIG): return r["target_match"]>=config["min_target_match"] and r["grounding"]>=config["min_grounding_edge"] and r["relation_score"]>=config["min_relation_score"]
def relation_lookup(rels): return {(r["source"],r["target"]):r for r in rels}
def select_formal_supports(rels,config=BAF_CONFIG): return sorted(set((r["source"],r["target"]) for r in rels if r["type"]=="SUPPORT" and r["target"]=="CLAIM" and r["source"].startswith("P") and relation_passes(r,config)))
def select_con_claim_attacks(rels,config=BAF_CONFIG):
    p=[r for r in rels if r["type"]=="ATTACK" and r["target"]=="CLAIM" and r["source"].startswith("N") and relation_passes(r,config)]
    p.sort(key=lambda x:x["relation_score"],reverse=True); return p[:config["max_claim_attacks"]]
def select_independent_conflict_pairs(rels,config=BAF_CONFIG):
    lk=relation_lookup(rels); cand=[]
    pro=sorted({r["source"] for r in rels if r["source"].startswith("P")}); con=sorted({r["source"] for r in rels if r["source"].startswith("N")})
    for p in pro:
        for n in con:
            pc,cp=lk.get((p,n)),lk.get((n,p))
            if not pc or not cp: continue
            if pc["type"]=="ATTACK" and cp["type"]=="ATTACK" and relation_passes(pc,config) and relation_passes(cp,config):
                cand.append({"pro":p,"con":n,"pair_score":min(pc["relation_score"],cp["relation_score"])})
    cand.sort(key=lambda x:x["pair_score"],reverse=True); sel=[]; up=set(); uc=set()
    for c in cand:
        if c["pro"] in up or c["con"] in uc: continue
        sel.append(c); up.add(c["pro"]); uc.add(c["con"])
        if len(sel)>=config["max_conflict_pairs"]: break
    return sel,cand
def support_closure(nodes,sup):
    cl=set(sup); ch=True
    while ch:
        ch=False; new=set()
        for a,b in cl:
            for c,d in cl:
                if b==c and (a,d) not in cl: new.add((a,d))
        if new: cl|=new; ch=True
    return cl
def derive_attacks(nodes,direct,sup):
    sc=support_closure(nodes,sup); sa=set(); se=set()
    for s,sd in sc:
        for at,t in direct:
            if sd==at: sa.add((s,t))
    for at,ad in direct:
        for s,sd in sc:
            if ad==s: se.add((at,sd))
    eff=(set(direct)|sa|se); eff={e for e in eff if e[0]!=e[1]}
    return {"support_closure":sorted(sc),"supported_attacks":sorted(sa),"secondary_attacks":sorted(se),"effective_attacks":sorted(eff)}
print("✅ CELL 4 ready")

✅ CELL 4 ready


In [7]:
def all_subsets(items):
    il=sorted(items)
    for sz in range(len(il)+1):
        for sub in combinations(il,sz): yield set(sub)
def cf(e,att): return not any(s in e and t in e for s,t in att)
def att_of(a,att): return {s for s,t in att if t==a}
def ext_att(e,a,att): return any(s in e and t==a for s,t in att)
def defends(e,a,att): return all(ext_att(e,x,att) for x in att_of(a,att))
def admissible(e,att): return cf(e,att) and all(defends(e,a,att) for a in e)
def preferred_ext(args,att):
    adm=[s for s in all_subsets(args) if admissible(s,att)]
    return [c for c in adm if not any(c<o for o in adm)]
def stable_ext(args,att):
    out=[]
    for c in all_subsets(args):
        if not cf(c,att): continue
        if all(ext_att(c,a,att) for a in args-c): out.append(c)
    return out
def norm(exts): return [sorted(e) for e in exts]
def decisions(exts,cid="CLAIM"):
    if not exts: return {"credulous":0,"skeptical":0,"majority":0}
    cc=sum(cid in e for e in exts)
    return {"credulous":int(cc>=1),"skeptical":int(cc==len(exts)),"majority":int(cc*2>=len(exts))}
def rebuild(inter,config=BAF_CONFIG):
    nodes=dict(inter["nodes"]); rels=inter["all_audited_relations"]; sup=set(select_formal_supports(rels,config))
    sca=select_con_claim_attacks(rels,config); sp,_=select_independent_conflict_pairs(rels,config); direct=set()
    for r in sca: direct.add((r["source"],r["target"]))
    for p in sp: direct.add((p["pro"],p["con"])); direct.add((p["con"],p["pro"]))
    conn={"CLAIM"}
    for st in sup|direct: conn.add(st[0]); conn.add(st[1])
    nodes={k:v for k,v in nodes.items() if k in conn}; vids=set(nodes)
    sup={e for e in sup if e[0] in vids and e[1] in vids}; direct={e for e in direct if e[0] in vids and e[1] in vids}
    der=derive_attacks(vids,direct,sup); eff=set(map(tuple,der["effective_attacks"]))
    pref=preferred_ext(vids,eff); stab=stable_ext(vids,eff)
    return {**inter,"nodes":nodes,"supports":sorted(sup),"selected_claim_attacks":sca,
        "selected_conflict_pairs":[{"pro":p["pro"],"con":p["con"],"pair_score":p["pair_score"]} for p in sp],
        "direct_attacks":sorted(direct),"support_closure":der["support_closure"],"supported_attacks":der["supported_attacks"],
        "secondary_attacks":der["secondary_attacks"],"effective_attacks":der["effective_attacks"],
        "preferred_extensions":norm(pref),"stable_extensions":norm(stab),"preferred_decisions":decisions(pref),"stable_decisions":decisions(stab)}
def run_v3_baf_example(row,config=BAF_CONFIG):
    claim=str(row["claim"]).strip(); facts=str(row["facts_text"]).strip(); gold=int(row["gold"])
    gen=generate_and_score_arguments(claim,facts,config); sel=select_arguments(gen,config); nodes=sel["nodes"]; nodes["CLAIM"]["text"]=claim
    audit=audit_baf_relations(claim,facts,nodes,config,config["audit_batch_size"])
    inter={"example_index":int(row["example_index"]),"claim":claim,"facts_text":facts,"gold":gold,"interpreted_claim":gen["interpreted_claim"],"generated_arguments":gen,"argument_scores":sel["all_scored"],"nodes":nodes,"all_audited_relations":audit["all_audited_relations"],"model":MODEL_NAME,"config":dict(config)}
    return rebuild(inter,config)
print("✅ CELL 5 ready — pipeline complete")

✅ CELL 5 ready — pipeline complete


In [18]:
CKPT = WORK_DIR / "gptoss_com2sense_ckpt.pkl"
def load_ck(): return pickle.load(open(CKPT,"rb")) if CKPT.exists() else {}
def save_ck(d): pickle.dump(d, open(CKPT,"wb"))
def run_batch(n=100):
    done = load_ck()
    todo = [i for i in range(len(baf_df)) if i not in done][:n]
    print(f"Have {len(done)}/{len(baf_df)}. Running {len(todo)} (GPT-OSS COM2SENSE)...")
    for k, idx in enumerate(todo, 1):
        row = baf_df.iloc[idx].copy()
        try:
            row["facts_text"] = generate_facts(row["claim"])
            res = run_v3_baf_example(row)
        except Exception as e:
            print(f"error at {idx}: {str(e)[:50]}"); continue
        d = load_ck(); d[idx] = res; save_ck(d)
        if k % 10 == 0 or k == len(todo):
            cur = load_ck()
            acc = np.mean([cur[i]["preferred_decisions"]["credulous"]==cur[i]["gold"] for i in cur])
            print(f"  {k}/{len(todo)} → {len(cur)} total | running acc {acc*100:.1f}%")
    print(f"✅ {len(load_ck())}/{len(baf_df)} done\n")

# RUN
run_batch(100)

# NET_SUPPORT CHECK
results = [load_ck()[k] for k in sorted(load_ck())]
gold = [r["gold"] for r in results]
def net_support_pred(r): return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))
from collections import Counter
print(f"=== GPT-OSS COM2SENSE at {len(results)} ===")
for rule in ["credulous","skeptical","majority"]:
    p=[r["preferred_decisions"][rule] for r in results]
    print(f"  Extension {rule}: {accuracy_score(gold,p)*100:.1f}%")
print(f"  net_support: {accuracy_score(gold,[net_support_pred(r) for r in results])*100:.1f}%")
print(f"  extension counts: {dict(sorted(Counter(len(r['preferred_extensions']) for r in results).items()))}")

# BACKUP (green button)
import base64, zipfile
from IPython.display import HTML, display
data=[load_ck()[k] for k in sorted(load_ck())]
N=len(data); pkl=WORK_DIR/f"gptoss_com2sense_{N}.pkl"; pickle.dump(data,open(pkl,"wb"))
zp=WORK_DIR/f"gptoss_com2sense_{N}_backup.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(pkl,arcname=f"gptoss_com2sense_{N}.pkl")
b64=base64.b64encode(open(zp,"rb").read()).decode()
display(HTML(f'<a download="gptoss_com2sense_{N}_backup.zip" href="data:application/zip;base64,{b64}" style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download gptoss_com2sense_{N}_backup.zip</a>'))
print(f"\nBacked up {N}")

Have 0/500. Running 100 (GPT-OSS COM2SENSE)...
  10/100 → 10 total | running acc 90.0%
  20/100 → 20 total | running acc 85.0%
error at 21: failed: Expecting ',' delimiter: line 101 column 6
  30/100 → 29 total | running acc 79.3%
  40/100 → 39 total | running acc 82.1%
  50/100 → 49 total | running acc 79.6%
  60/100 → 59 total | running acc 78.0%
  70/100 → 69 total | running acc 78.3%
  80/100 → 79 total | running acc 78.5%
  90/100 → 89 total | running acc 78.7%
error at 93: failed: Expecting ',' delimiter: line 92 column 6 
  100/100 → 98 total | running acc 78.6%
✅ 98/500 done

=== GPT-OSS COM2SENSE at 98 ===
  Extension credulous: 78.6%
  Extension skeptical: 75.5%
  Extension majority: 75.5%
  net_support: 79.6%
  extension counts: {1: 71, 2: 3, 4: 8, 8: 16}



Backed up 98


The Kaggle session was restarted after 98 claims; the next cells reload the saved claims from a Kaggle dataset before the run continues.

In [7]:
CKPT = WORK_DIR / "gptoss_com2sense_ckpt.pkl"
def load_ck(): return pickle.load(open(CKPT,"rb")) if CKPT.exists() else {}
def save_ck(d): pickle.dump(d, open(CKPT,"wb"))
def run_batch(n=100):
    done = load_ck()
    todo = [i for i in range(len(baf_df)) if i not in done][:n]
    print(f"Have {len(done)}/{len(baf_df)}. Running {len(todo)} (GPT-OSS COM2SENSE)...")
    for k, idx in enumerate(todo, 1):
        row = baf_df.iloc[idx].copy()
        try:
            row["facts_text"] = generate_facts(row["claim"])
            res = run_v3_baf_example(row)
        except Exception as e:
            print(f"error at {idx}: {str(e)[:50]}"); continue
        d = load_ck(); d[idx] = res; save_ck(d)
        if k % 10 == 0 or k == len(todo):
            cur = load_ck()
            acc = np.mean([cur[i]["preferred_decisions"]["credulous"]==cur[i]["gold"] for i in cur])
            print(f"  {k}/{len(todo)} → {len(cur)} total | running acc {acc*100:.1f}%")
    print(f"✅ {len(load_ck())}/{len(baf_df)} done")
print("✅ runner ready")

✅ runner ready


In [9]:
import glob
all_pkl = glob.glob("/kaggle/input/**/*.pkl", recursive=True)
print("All .pkl files in input:")
for f in all_pkl: print(" ", f)

All .pkl files in input:
  /kaggle/input/gptoss-com2sense-98-backup/gptoss_com2sense_98.pkl


In [11]:
path = "/kaggle/input/gptoss-com2sense-98-backup/gptoss_com2sense_98.pkl"
data = pickle.load(open(path, "rb"))
print("loaded:", len(data), "items | keys:", list(data[0].keys())[:5])
restored = {r["example_index"]: r for r in data}
save_ck(restored)
print(f"Restored {len(load_ck())} examples")   # should say 98

loaded: 98 items | keys: ['example_index', 'claim', 'facts_text', 'gold', 'interpreted_claim']
Restored 98 examples


In [12]:
results = [load_ck()[k] for k in sorted(load_ck())]
gold = [r["gold"] for r in results]
n = len(results)
from collections import Counter
def net_support_pred(r): return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))
def no_objection_pred(r, ek):
    exts=r[ek]
    if not exts: return 0
    eff=set(map(tuple,r["effective_attacks"]))
    for ext in exts:
        es=set(ext)
        if "CLAIM" in es and not any(t=="CLAIM" and s in es for (s,t) in eff): return 1
    return 0
def mrow(name, pred):
    tn,fp,fn,tp=confusion_matrix(gold,pred,labels=[0,1]).ravel()
    return {"method":name,"accuracy":round(accuracy_score(gold,pred)*100,1),"precision":round(precision_score(gold,pred,zero_division=0)*100,1),"recall":round(recall_score(gold,pred,zero_division=0)*100,1),"f1":round(f1_score(gold,pred,zero_division=0)*100,1),"TN":tn,"FP":fp,"FN":fn,"TP":tp}
rows=[]
for rule in ["credulous","skeptical","majority"]:
    rows.append(mrow(f"preferred_{rule}",[r["preferred_decisions"][rule] for r in results]))
rows.append(mrow("preferred_no_objection",[no_objection_pred(r,"preferred_extensions") for r in results]))
for rule in ["credulous","skeptical","majority"]:
    rows.append(mrow(f"stable_{rule}",[r["stable_decisions"][rule] for r in results]))
rows.append(mrow("stable_no_objection",[no_objection_pred(r,"stable_extensions") for r in results]))
rows.append(mrow("net_support",[net_support_pred(r) for r in results]))
table=pd.DataFrame(rows).sort_values("accuracy",ascending=False).reset_index(drop=True)
pd.set_option("display.max_columns",None); pd.set_option("display.width",220)
print(f"=== GPT-OSS COM2SENSE — ALL semantics ({n}) ===\n"); print(table.to_string())
print(f"\nextension counts: {dict(sorted(Counter(len(r['preferred_extensions']) for r in results).items()))}")
print(f"gold balance: True={sum(gold)} False={n-sum(gold)}")

=== GPT-OSS COM2SENSE — ALL semantics (98) ===

                   method  accuracy  precision  recall    f1  TN  FP  FN  TP
0             net_support      79.6       70.3    97.8  81.8  33  19   1  45
1     preferred_credulous      78.6       71.9    89.1  79.6  36  16   5  41
2        stable_credulous      78.6       71.9    89.1  79.6  36  16   5  41
3     stable_no_objection      78.6       71.9    89.1  79.6  36  16   5  41
4  preferred_no_objection      78.6       71.9    89.1  79.6  36  16   5  41
5     preferred_skeptical      75.5       80.6    63.0  70.7  45   7  17  29
6      preferred_majority      75.5       80.6    63.0  70.7  45   7  17  29
7         stable_majority      75.5       80.6    63.0  70.7  45   7  17  29
8        stable_skeptical      75.5       80.6    63.0  70.7  45   7  17  29

extension counts: {1: 71, 2: 3, 4: 8, 8: 16}
gold balance: True=46 False=52


In [14]:
run_batch(100)   # → ~200

Have 98/500. Running 100 (GPT-OSS COM2SENSE)...
error at 93: failed: Expecting ',' delimiter: line 29 column 6 
error at 104: failed: Expecting ',' delimiter: line 47 column 6 
error at 109: failed: Expecting ',' delimiter: line 110 column 6
  20/100 → 115 total | running acc 77.4%
  30/100 → 125 total | running acc 78.4%
error at 134: failed: Expecting ',' delimiter: line 47 column 6 
  40/100 → 134 total | running acc 77.6%
  50/100 → 144 total | running acc 77.8%
  60/100 → 154 total | running acc 77.9%
  70/100 → 164 total | running acc 77.4%
error at 174: failed: Expecting ',' delimiter: line 83 column 6 
  80/100 → 173 total | running acc 78.0%
  90/100 → 183 total | running acc 77.0%
  100/100 → 193 total | running acc 76.7%
✅ 193/500 done


In [15]:
results = [load_ck()[k] for k in sorted(load_ck())]
gold = [r["gold"] for r in results]
n = len(results)
from collections import Counter
def net_support_pred(r): return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))
def no_objection_pred(r, ek):
    exts=r[ek]
    if not exts: return 0
    eff=set(map(tuple,r["effective_attacks"]))
    for ext in exts:
        es=set(ext)
        if "CLAIM" in es and not any(t=="CLAIM" and s in es for (s,t) in eff): return 1
    return 0
def mrow(name, pred):
    tn,fp,fn,tp=confusion_matrix(gold,pred,labels=[0,1]).ravel()
    return {"method":name,"accuracy":round(accuracy_score(gold,pred)*100,1),"precision":round(precision_score(gold,pred,zero_division=0)*100,1),"recall":round(recall_score(gold,pred,zero_division=0)*100,1),"f1":round(f1_score(gold,pred,zero_division=0)*100,1),"TN":tn,"FP":fp,"FN":fn,"TP":tp}
rows=[]
for rule in ["credulous","skeptical","majority"]:
    rows.append(mrow(f"preferred_{rule}",[r["preferred_decisions"][rule] for r in results]))
rows.append(mrow("preferred_no_objection",[no_objection_pred(r,"preferred_extensions") for r in results]))
for rule in ["credulous","skeptical","majority"]:
    rows.append(mrow(f"stable_{rule}",[r["stable_decisions"][rule] for r in results]))
rows.append(mrow("stable_no_objection",[no_objection_pred(r,"stable_extensions") for r in results]))
rows.append(mrow("net_support",[net_support_pred(r) for r in results]))
table=pd.DataFrame(rows).sort_values("accuracy",ascending=False).reset_index(drop=True)
pd.set_option("display.max_columns",None); pd.set_option("display.width",220)
print(f"=== GPT-OSS COM2SENSE — ALL semantics ({n}) ===\n"); print(table.to_string())
print(f"\nextension counts: {dict(sorted(Counter(len(r['preferred_extensions']) for r in results).items()))}")
print(f"gold balance: True={sum(gold)} False={n-sum(gold)}")

import base64, zipfile
from IPython.display import HTML, display
data=[load_ck()[k] for k in sorted(load_ck())]
N=len(data); pkl=WORK_DIR/f"gptoss_com2sense_{N}.pkl"; pickle.dump(data,open(pkl,"wb"))
zp=WORK_DIR/f"gptoss_com2sense_{N}_backup.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(pkl,arcname=f"gptoss_com2sense_{N}.pkl")
b64=base64.b64encode(open(zp,"rb").read()).decode()
display(HTML(f'<a download="gptoss_com2sense_{N}_backup.zip" href="data:application/zip;base64,{b64}" style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download gptoss_com2sense_{N}_backup.zip</a>'))
print(f"Backed up {N}")

=== GPT-OSS COM2SENSE — ALL semantics (193) ===

                   method  accuracy  precision  recall    f1  TN  FP  FN  TP
0             net_support      78.2       71.5    97.0  82.4  53  39   3  98
1     preferred_credulous      76.7       72.2    90.1  80.2  57  35  10  91
2        stable_credulous      76.7       72.2    90.1  80.2  57  35  10  91
3     stable_no_objection      76.7       72.2    90.1  80.2  57  35  10  91
4  preferred_no_objection      76.7       72.2    90.1  80.2  57  35  10  91
5         stable_majority      70.5       76.8    62.4  68.9  73  19  38  63
6      preferred_majority      70.5       76.8    62.4  68.9  73  19  38  63
7     preferred_skeptical      69.4       76.2    60.4  67.4  73  19  40  61
8        stable_skeptical      69.4       76.2    60.4  67.4  73  19  40  61

extension counts: {1: 138, 2: 8, 4: 22, 8: 25}
gold balance: True=101 False=92


Backed up 193


In [16]:
# RUN next 100
run_batch(100)

# FULL TABLE
results = [load_ck()[k] for k in sorted(load_ck())]
gold = [r["gold"] for r in results]; n = len(results)
from collections import Counter
def net_support_pred(r): return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))
def mrow(name, pred):
    tn,fp,fn,tp=confusion_matrix(gold,pred,labels=[0,1]).ravel()
    return {"method":name,"accuracy":round(accuracy_score(gold,pred)*100,1),"f1":round(f1_score(gold,pred,zero_division=0)*100,1),"TN":tn,"FP":fp,"FN":fn,"TP":tp}
rows=[]
for rule in ["credulous","skeptical","majority"]:
    rows.append(mrow(f"preferred_{rule}",[r["preferred_decisions"][rule] for r in results]))
rows.append(mrow("net_support",[net_support_pred(r) for r in results]))
table=pd.DataFrame(rows).sort_values("accuracy",ascending=False).reset_index(drop=True)
pd.set_option("display.max_columns",None); pd.set_option("display.width",200)
print(f"\n=== GPT-OSS COM2SENSE — semantics ({n}) ===\n"); print(table.to_string())
print(f"\nextension counts: {dict(sorted(Counter(len(r['preferred_extensions']) for r in results).items()))}")
print(f"gold balance: True={sum(gold)} False={n-sum(gold)}")

# BACKUP
import base64, zipfile
from IPython.display import HTML, display
data=[load_ck()[k] for k in sorted(load_ck())]
N=len(data); pkl=WORK_DIR/f"gptoss_com2sense_{N}.pkl"; pickle.dump(data,open(pkl,"wb"))
zp=WORK_DIR/f"gptoss_com2sense_{N}_backup.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(pkl,arcname=f"gptoss_com2sense_{N}.pkl")
b64=base64.b64encode(open(zp,"rb").read()).decode()
display(HTML(f'<a download="gptoss_com2sense_{N}_backup.zip" href="data:application/zip;base64,{b64}" style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download gptoss_com2sense_{N}_backup.zip</a>'))
print(f"Backed up {N}")

Have 193/500. Running 100 (GPT-OSS COM2SENSE)...
error at 200: failed: Expecting ',' delimiter: line 146 column 6
  10/100 → 202 total | running acc 76.2%
  20/100 → 212 total | running acc 76.9%
error at 213: failed: Expecting ',' delimiter: line 137 column 6
  30/100 → 221 total | running acc 76.5%
  40/100 → 231 total | running acc 76.6%
error at 233: failed: Expecting ',' delimiter: line 128 column 6
  50/100 → 240 total | running acc 77.1%
  60/100 → 250 total | running acc 76.8%
  70/100 → 260 total | running acc 76.2%
error at 264: failed: Expecting ',' delimiter: line 7 column 24 
error at 268: failed: Expecting ',' delimiter: line 119 column 6
error at 270: failed: Expecting ',' delimiter: line 110 column 6
  80/100 → 267 total | running acc 76.0%
error at 280: failed: Expecting ',' delimiter: line 110 column 6
  90/100 → 276 total | running acc 75.0%
  100/100 → 286 total | running acc 74.5%
✅ 286/500 done

=== GPT-OSS COM2SENSE — semantics (286) ===

                method  

Backed up 286


In [18]:
results = [load_ck()[k] for k in sorted(load_ck())]
gold = [r["gold"] for r in results]
n = len(results)
from collections import Counter
def net_support_pred(r): return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))
def no_objection_pred(r, ek):
    exts=r[ek]
    if not exts: return 0
    eff=set(map(tuple,r["effective_attacks"]))
    for ext in exts:
        es=set(ext)
        if "CLAIM" in es and not any(t=="CLAIM" and s in es for (s,t) in eff): return 1
    return 0
def mrow(name, pred):
    tn,fp,fn,tp=confusion_matrix(gold,pred,labels=[0,1]).ravel()
    return {"method":name,"accuracy":round(accuracy_score(gold,pred)*100,1),"precision":round(precision_score(gold,pred,zero_division=0)*100,1),"recall":round(recall_score(gold,pred,zero_division=0)*100,1),"f1":round(f1_score(gold,pred,zero_division=0)*100,1),"TN":tn,"FP":fp,"FN":fn,"TP":tp}
rows=[]
for rule in ["credulous","skeptical","majority"]:
    rows.append(mrow(f"preferred_{rule}",[r["preferred_decisions"][rule] for r in results]))
rows.append(mrow("preferred_no_objection",[no_objection_pred(r,"preferred_extensions") for r in results]))
for rule in ["credulous","skeptical","majority"]:
    rows.append(mrow(f"stable_{rule}",[r["stable_decisions"][rule] for r in results]))
rows.append(mrow("stable_no_objection",[no_objection_pred(r,"stable_extensions") for r in results]))
rows.append(mrow("net_support",[net_support_pred(r) for r in results]))
table=pd.DataFrame(rows).sort_values("accuracy",ascending=False).reset_index(drop=True)
pd.set_option("display.max_columns",None); pd.set_option("display.width",220)
print(f"=== GPT-OSS COM2SENSE — ALL semantics ({n}) ===\n"); print(table.to_string())
print(f"\nextension counts: {dict(sorted(Counter(len(r['preferred_extensions']) for r in results).items()))}")
print(f"gold balance: True={sum(gold)} False={n-sum(gold)}")
print(f"\nFor comparison: DF-QuAD 0.5 Base 52.2% | Est. Base 73.8% (500) | Direct 76% (100)")

import base64, zipfile
from IPython.display import HTML, display
data=[load_ck()[k] for k in sorted(load_ck())]
N=len(data); pkl=WORK_DIR/f"gptoss_com2sense_{N}.pkl"; pickle.dump(data,open(pkl,"wb"))
zp=WORK_DIR/f"gptoss_com2sense_{N}_backup.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(pkl,arcname=f"gptoss_com2sense_{N}.pkl")
b64=base64.b64encode(open(zp,"rb").read()).decode()
display(HTML(f'<a download="gptoss_com2sense_{N}_backup.zip" href="data:application/zip;base64,{b64}" style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download gptoss_com2sense_{N}_backup.zip</a>'))
print(f"Backed up {N}")

=== GPT-OSS COM2SENSE — ALL semantics (286) ===

                   method  accuracy  precision  recall    f1   TN  FP  FN   TP
0             net_support      75.5       68.3    96.6  80.0   76  65   5  140
1     preferred_credulous      74.5       68.9    90.3  78.2   82  59  14  131
2        stable_credulous      74.5       68.9    90.3  78.2   82  59  14  131
3     stable_no_objection      74.5       68.9    90.3  78.2   82  59  14  131
4  preferred_no_objection      74.5       68.9    90.3  78.2   82  59  14  131
5         stable_majority      71.0       74.6    64.8  69.4  109  32  51   94
6      preferred_majority      71.0       74.6    64.8  69.4  109  32  51   94
7     preferred_skeptical      70.6       74.8    63.4  68.7  110  31  53   92
8        stable_skeptical      70.6       74.8    63.4  68.7  110  31  53   92

extension counts: {1: 207, 2: 11, 4: 28, 8: 40}
gold balance: True=145 False=141

For comparison: DF-QuAD 0.5 Base 52.2% | Est. Base 73.8% (500) | Direct 76% (1

Backed up 286


The session was restarted again after 286 claims; the saved claims are reloaded and the run continues to 474 claims (the remaining claims failed after all retries; Table 1, footnote e).

In [8]:
CKPT = WORK_DIR / "gptoss_com2sense_ckpt.pkl"
def load_ck(): return pickle.load(open(CKPT,"rb")) if CKPT.exists() else {}
def save_ck(d): pickle.dump(d, open(CKPT,"wb"))
def run_batch(n=100):
    done = load_ck()
    todo = [i for i in range(len(baf_df)) if i not in done][:n]
    print(f"Have {len(done)}/{len(baf_df)}. Running {len(todo)} (GPT-OSS COM2SENSE)...")
    for k, idx in enumerate(todo, 1):
        row = baf_df.iloc[idx].copy()
        try:
            row["facts_text"] = generate_facts(row["claim"])
            res = run_v3_baf_example(row)
        except Exception as e:
            print(f"error at {idx}: {str(e)[:50]}"); continue
        d = load_ck(); d[idx] = res; save_ck(d)
        if k % 10 == 0 or k == len(todo):
            cur = load_ck()
            acc = np.mean([cur[i]["preferred_decisions"]["credulous"]==cur[i]["gold"] for i in cur])
            print(f"  {k}/{len(todo)} → {len(cur)} total | running acc {acc*100:.1f}%")
    print(f"✅ {len(load_ck())}/{len(baf_df)} done")
print("✅ runner ready")

✅ runner ready


In [9]:
import glob
hits = glob.glob("/kaggle/input/**/gptoss_com2sense_286.pkl", recursive=True)
print("found:", hits)
data = pickle.load(open(hits[0], "rb"))
restored = {r["example_index"]: r for r in data}
save_ck(restored)
print(f"Restored {len(load_ck())} examples")   # should say 286

found: ['/kaggle/input/gptoss-com2sense-286-backup/gptoss_com2sense_286.pkl']
Restored 286 examples


In [11]:
def run_batch_safe(n=100):
    done = load_ck()
    todo = [i for i in range(len(baf_df)) if i not in done][:n]
    print(f"Have {len(done)}/{len(baf_df)}. Running {len(todo)}...")
    for k, idx in enumerate(todo, 1):
        row = baf_df.iloc[idx].copy()
        try:
            row["facts_text"] = generate_facts(row["claim"])
            res = run_v3_baf_example(row)
        except Exception as e:
            print(f"error at {idx}: {str(e)[:50]}"); continue
        d = load_ck(); d[idx] = res; save_ck(d)
        cur_n = len(load_ck())
        if k % 10 == 0: print(f"  {k}/{len(todo)} → {cur_n} total")
        # auto-backup every 50 total
        if cur_n % 50 == 0:
            import base64, zipfile
            data=[load_ck()[kk] for kk in sorted(load_ck())]
            pkl=WORK_DIR/f"gptoss_com2sense_{cur_n}.pkl"; pickle.dump(data,open(pkl,"wb"))
            print(f"    💾 auto-saved at {cur_n} (download below when convenient)")
    print(f"✅ {len(load_ck())}/{len(baf_df)} done")

run_batch_safe(100)

Have 286/500. Running 100...
  10/100 → 296 total
    💾 auto-saved at 300 (download below when convenient)
  20/100 → 306 total
error at 311: failed: Expecting ',' delimiter: line 83 column 6 
  30/100 → 315 total
error at 321: failed: Expecting ',' delimiter: line 92 column 6 
  40/100 → 324 total
  50/100 → 334 total
  60/100 → 344 total
error at 348: failed: Expecting ',' delimiter: line 74 column 6 
    💾 auto-saved at 350 (download below when convenient)
  70/100 → 353 total
error at 359: failed: Expecting ',' delimiter: line 128 column 6
  80/100 → 362 total
error at 374: failed: empty
  90/100 → 371 total
  100/100 → 381 total
✅ 381/500 done


In [12]:
results = [load_ck()[k] for k in sorted(load_ck())]
gold = [r["gold"] for r in results]; n = len(results)
from collections import Counter
def net_support_pred(r): return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))
def no_objection_pred(r, ek):
    exts=r[ek]
    if not exts: return 0
    eff=set(map(tuple,r["effective_attacks"]))
    for ext in exts:
        es=set(ext)
        if "CLAIM" in es and not any(t=="CLAIM" and s in es for (s,t) in eff): return 1
    return 0
def mrow(name, pred):
    tn,fp,fn,tp=confusion_matrix(gold,pred,labels=[0,1]).ravel()
    return {"method":name,"accuracy":round(accuracy_score(gold,pred)*100,1),"precision":round(precision_score(gold,pred,zero_division=0)*100,1),"recall":round(recall_score(gold,pred,zero_division=0)*100,1),"f1":round(f1_score(gold,pred,zero_division=0)*100,1),"TN":tn,"FP":fp,"FN":fn,"TP":tp}
rows=[]
for rule in ["credulous","skeptical","majority"]:
    rows.append(mrow(f"preferred_{rule}",[r["preferred_decisions"][rule] for r in results]))
rows.append(mrow("preferred_no_objection",[no_objection_pred(r,"preferred_extensions") for r in results]))
for rule in ["credulous","skeptical","majority"]:
    rows.append(mrow(f"stable_{rule}",[r["stable_decisions"][rule] for r in results]))
rows.append(mrow("stable_no_objection",[no_objection_pred(r,"stable_extensions") for r in results]))
rows.append(mrow("net_support",[net_support_pred(r) for r in results]))
table=pd.DataFrame(rows).sort_values("accuracy",ascending=False).reset_index(drop=True)
pd.set_option("display.max_columns",None); pd.set_option("display.width",220)
print(f"=== GPT-OSS COM2SENSE — ALL semantics ({n}) ===\n"); print(table.to_string())
print(f"\nextension counts: {dict(sorted(Counter(len(r['preferred_extensions']) for r in results).items()))}")
print(f"gold balance: True={sum(gold)} False={n-sum(gold)}")

import base64, zipfile
from IPython.display import HTML, display
data=[load_ck()[k] for k in sorted(load_ck())]
N=len(data); pkl=WORK_DIR/f"gptoss_com2sense_{N}.pkl"; pickle.dump(data,open(pkl,"wb"))
zp=WORK_DIR/f"gptoss_com2sense_{N}_backup.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(pkl,arcname=f"gptoss_com2sense_{N}.pkl")
b64=base64.b64encode(open(zp,"rb").read()).decode()
display(HTML(f'<a download="gptoss_com2sense_{N}_backup.zip" href="data:application/zip;base64,{b64}" style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download gptoss_com2sense_{N}_backup.zip</a>'))
print(f"Backed up {N}")

=== GPT-OSS COM2SENSE — ALL semantics (381) ===

                   method  accuracy  precision  recall    f1   TN  FP  FN   TP
0             net_support      75.9       69.1    95.9  80.3  101  84   8  188
1     preferred_credulous      74.8       69.8    89.8  78.6  109  76  20  176
2        stable_credulous      74.8       69.8    89.8  78.6  109  76  20  176
3     stable_no_objection      74.8       69.8    89.8  78.6  109  76  20  176
4  preferred_no_objection      74.8       69.8    89.8  78.6  109  76  20  176
5     preferred_skeptical      73.8       78.6    67.3  72.5  149  36  64  132
6      preferred_majority      73.8       77.9    68.4  72.8  147  38  62  134
7         stable_majority      73.8       77.9    68.4  72.8  147  38  62  134
8        stable_skeptical      73.8       78.6    67.3  72.5  149  36  64  132

extension counts: {1: 280, 2: 14, 4: 37, 8: 50}
gold balance: True=196 False=185


Backed up 381


In [13]:
def run_batch_safe(n=100):
    done = load_ck()
    todo = [i for i in range(len(baf_df)) if i not in done][:n]
    print(f"Have {len(done)}/{len(baf_df)}. Running {len(todo)}...")
    for k, idx in enumerate(todo, 1):
        row = baf_df.iloc[idx].copy()
        try:
            row["facts_text"] = generate_facts(row["claim"])
            res = run_v3_baf_example(row)
        except Exception as e:
            print(f"error at {idx}: {str(e)[:50]}"); continue
        d = load_ck(); d[idx] = res; save_ck(d)
        cur_n = len(load_ck())
        if k % 10 == 0: print(f"  {k}/{len(todo)} → {cur_n} total")
        if cur_n % 50 == 0:
            data=[load_ck()[kk] for kk in sorted(load_ck())]
            pickle.dump(data, open(WORK_DIR/f"gptoss_com2sense_{cur_n}.pkl","wb"))
            print(f"    💾 auto-saved at {cur_n}")
    print(f"✅ {len(load_ck())}/{len(baf_df)} done\n")

run_batch_safe(100)

results = [load_ck()[k] for k in sorted(load_ck())]
gold = [r["gold"] for r in results]; n = len(results)
from collections import Counter
def net_support_pred(r): return int(len(r["supports"]) >= sum(1 for (s,t) in r["effective_attacks"] if t=="CLAIM"))
def no_objection_pred(r, ek):
    exts=r[ek]
    if not exts: return 0
    eff=set(map(tuple,r["effective_attacks"]))
    for ext in exts:
        es=set(ext)
        if "CLAIM" in es and not any(t=="CLAIM" and s in es for (s,t) in eff): return 1
    return 0
def mrow(name, pred):
    tn,fp,fn,tp=confusion_matrix(gold,pred,labels=[0,1]).ravel()
    return {"method":name,"accuracy":round(accuracy_score(gold,pred)*100,1),"precision":round(precision_score(gold,pred,zero_division=0)*100,1),"recall":round(recall_score(gold,pred,zero_division=0)*100,1),"f1":round(f1_score(gold,pred,zero_division=0)*100,1),"TN":tn,"FP":fp,"FN":fn,"TP":tp}
rows=[]
for rule in ["credulous","skeptical","majority"]:
    rows.append(mrow(f"preferred_{rule}",[r["preferred_decisions"][rule] for r in results]))
rows.append(mrow("preferred_no_objection",[no_objection_pred(r,"preferred_extensions") for r in results]))
for rule in ["credulous","skeptical","majority"]:
    rows.append(mrow(f"stable_{rule}",[r["stable_decisions"][rule] for r in results]))
rows.append(mrow("stable_no_objection",[no_objection_pred(r,"stable_extensions") for r in results]))
rows.append(mrow("net_support",[net_support_pred(r) for r in results]))
table=pd.DataFrame(rows).sort_values("accuracy",ascending=False).reset_index(drop=True)
pd.set_option("display.max_columns",None); pd.set_option("display.width",220)
print(f"=== GPT-OSS COM2SENSE — ALL semantics ({n}) ===\n"); print(table.to_string())
print(f"\nextension counts: {dict(sorted(Counter(len(r['preferred_extensions']) for r in results).items()))}")
print(f"gold balance: True={sum(gold)} False={n-sum(gold)}")

import base64, zipfile
from IPython.display import HTML, display
data=[load_ck()[k] for k in sorted(load_ck())]
N=len(data); pkl=WORK_DIR/f"gptoss_com2sense_{N}.pkl"; pickle.dump(data,open(pkl,"wb"))
zp=WORK_DIR/f"gptoss_com2sense_{N}_backup.zip"
with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(pkl,arcname=f"gptoss_com2sense_{N}.pkl")
b64=base64.b64encode(open(zp,"rb").read()).decode()
display(HTML(f'<a download="gptoss_com2sense_{N}_backup.zip" href="data:application/zip;base64,{b64}" style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download gptoss_com2sense_{N}_backup.zip</a>'))
print(f"Backed up {N}")

Have 381/500. Running 100...
  10/100 → 391 total
    💾 auto-saved at 400
  20/100 → 401 total
  30/100 → 411 total
  40/100 → 421 total
error at 426: failed: Expecting ',' delimiter: line 92 column 6 
error at 430: failed: Expecting ',' delimiter: line 29 column 6 
error at 431: failed: Expecting ',' delimiter: line 110 column 6
error at 435: failed: Expecting ',' delimiter: line 101 column 6
error at 436: failed: Expecting ',' delimiter: line 83 column 6 
  60/100 → 436 total
error at 442: failed: Expecting ',' delimiter: line 128 column 6
    💾 auto-saved at 450
error at 458: failed: Expecting ',' delimiter: line 92 column 6 
  80/100 → 454 total
  90/100 → 464 total
  100/100 → 474 total
✅ 474/500 done

=== GPT-OSS COM2SENSE — ALL semantics (474) ===

                   method  accuracy  precision  recall    f1   TN   FP  FN   TP
0             net_support      75.3       68.0    96.2  79.7  128  108   9  229
1     preferred_credulous      74.3       69.0    88.7  77.6  141   95  27

Backed up 474


Final results on the 474 saved frameworks (no API calls).

In [1]:
import os, glob, pickle, zipfile
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)

IN = "/kaggle/input"
OUT = "/kaggle/working/unzipped"
os.makedirs(OUT, exist_ok=True)

# unzip any zip found in the input
for z in glob.glob(IN + "/**/*.zip", recursive=True):
    zipfile.ZipFile(z).extractall(OUT)
    print("unzipped:", z)

pkls = (glob.glob(IN + "/**/*.pkl", recursive=True)
        + glob.glob(OUT + "/**/*.pkl", recursive=True))
print("pkl files found:")
for p in pkls:
    print("  ", p, round(os.path.getsize(p) / 1e6, 1), "MB")


def to_list(obj):
    if isinstance(obj, dict):
        return [obj[k] for k in sorted(obj)]
    return list(obj)


best = None
for p in pkls:
    try:
        rows = to_list(pickle.load(open(p, "rb")))
        if rows and isinstance(rows[0], dict) and "gold" in rows[0]:
            if best is None or len(rows) > len(best[1]):
                best = (p, rows)
    except Exception as e:
        print("  skip", p, str(e)[:60])

PATH, results = best
print("\nusing:", PATH)
print("claims:", len(results), "| true:", sum(r["gold"] for r in results))
print("keys:", sorted(results[0].keys())[:25])

pkl files found:
   /kaggle/input/gptoss-com2sense-286-backup/gptoss_com2sense_286.pkl 1.5 MB
   /kaggle/input/gptoss-com2sense-98-backup/gptoss_com2sense_98.pkl 0.5 MB
   /kaggle/input/gptoss-com2sense-474-backup/gptoss_com2sense_474.pkl 2.5 MB

using: /kaggle/input/gptoss-com2sense-474-backup/gptoss_com2sense_474.pkl
claims: 474 | true: 238
keys: ['all_audited_relations', 'argument_scores', 'claim', 'config', 'direct_attacks', 'effective_attacks', 'example_index', 'facts_text', 'generated_arguments', 'gold', 'interpreted_claim', 'model', 'nodes', 'preferred_decisions', 'preferred_extensions', 'secondary_attacks', 'selected_claim_attacks', 'selected_conflict_pairs', 'stable_decisions', 'stable_extensions', 'support_closure', 'supported_attacks', 'supports']


In [2]:
def decide(exts, cid="CLAIM"):
    if not exts:
        return {"credulous": 0, "skeptical": 0, "majority": 0}
    cc = sum(cid in e for e in exts)
    return {"credulous": int(cc >= 1),
            "skeptical": int(cc == len(exts)),
            "majority": int(cc * 2 >= len(exts))}


def no_objection(r, key):
    exts = [set(e) for e in r[key]]
    if not exts or not any("CLAIM" in e for e in exts):
        return 0
    att = {s for (s, t) in map(tuple, r["direct_attacks"])
           if t == "CLAIM"}
    return int(not (att & set().union(*exts)))


def net_support(r):
    n_att = sum(1 for (s, t) in map(tuple, r["effective_attacks"])
                if t == "CLAIM")
    return int(len(r["supports"]) >= n_att)


gold = [int(r["gold"]) for r in results]
n = len(results)


def row(name, pred):
    tn, fp, fn, tp = confusion_matrix(gold, pred, labels=[0, 1]).ravel()
    return {"method": name, "n": n,
            "acc": round(accuracy_score(gold, pred) * 100, 1),
            "prec": round(precision_score(gold, pred,
                                          zero_division=0) * 100, 1),
            "rec": round(recall_score(gold, pred,
                                      zero_division=0) * 100, 1),
            "f1": round(f1_score(gold, pred, zero_division=0) * 100, 1),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}


rows = []
for sem in ["preferred", "stable"]:
    key = f"{sem}_extensions"
    dec = [decide([set(e) for e in r[key]]) for r in results]
    for rule in ["credulous", "skeptical", "majority"]:
        rows.append(row(f"{sem}_{rule}", [d[rule] for d in dec]))
    rows.append(row(f"{sem}_no_objection",
                    [no_objection(r, key) for r in results]))
rows.append(row("net_support", [net_support(r) for r in results]))

pd.set_option("display.width", 220)
print(f"=== GPT-OSS COM2SENSE - Version 2 ({n} claims) ===\n")
print(pd.DataFrame(rows).to_string(index=False))

# ----- extensions -----
n_ext = [len(r["preferred_extensions"]) for r in results]
cnt = Counter(n_ext)
print("\nextension counts:", dict(sorted(cnt.items())))
print("mean extensions:", round(np.mean(n_ext), 2))
multi = sum(k > 1 for k in n_ext)
print(f"more than 1 extension: {multi} ({multi / n * 100:.1f}%)")

ext_rows = []
for k in sorted(cnt):
    sub = [r for r, e in zip(results, n_ext) if e == k]
    g = [int(r["gold"]) for r in sub]
    d = [decide([set(e) for e in r["preferred_extensions"]])
         for r in sub]
    ext_rows.append({
        "n_ext": k, "claims": len(sub),
        "percent": round(len(sub) / n * 100, 1),
        "credulous_acc": round(np.mean(
            [x["credulous"] == y for x, y in zip(d, g)]) * 100, 1),
        "skeptical_acc": round(np.mean(
            [x["skeptical"] == y for x, y in zip(d, g)]) * 100, 1)})
print(pd.DataFrame(ext_rows).to_string(index=False))

=== GPT-OSS COM2SENSE - Version 2 (474 claims) ===

                method   n  acc  prec  rec   f1  TN  FP  FN  TP
   preferred_credulous 474 74.3  69.0 88.7 77.6 141  95  27 211
   preferred_skeptical 474 73.2  77.3 66.0 71.2 190  46  81 157
    preferred_majority 474 73.2  76.8 66.8 71.5 188  48  79 159
preferred_no_objection 474 73.4  77.5 66.4 71.5 190  46  80 158
      stable_credulous 474 74.3  69.0 88.7 77.6 141  95  27 211
      stable_skeptical 474 73.2  77.3 66.0 71.2 190  46  81 157
       stable_majority 474 73.2  76.8 66.8 71.5 188  48  79 159
   stable_no_objection 474 73.4  77.5 66.4 71.5 190  46  80 158
           net_support 474 75.3  68.0 96.2 79.7 128 108   9 229

extension counts: {1: 343, 2: 21, 4: 51, 8: 59}
mean extensions: 2.24
more than 1 extension: 131 (27.6%)
 n_ext  claims  percent  credulous_acc  skeptical_acc
     1     343     72.4           83.1           83.1
     2      21      4.4           47.6           47.6
     4      51     10.8           56.9  